## Load EmoBank Dataset

In [5]:
import requests
import pandas as pd
from pathlib import Path

In [7]:
# Download EmoBank dataset if not already present
def download_emobank():
    """Download the EmoBank dataset from the official source."""
    url = "https://github.com/JULIELab/EmoBank/raw/master/corpus/emobank.csv"
    
    # Save to current working directory
    file_path = Path("emobank.csv")
    
    if not file_path.exists():
        print("Downloading EmoBank dataset...")
        response = requests.get(url)
        response.raise_for_status()
        
        with open(file_path, 'w', encoding='utf-8') as f:
            f.write(response.text)
        print(f"Downloaded EmoBank dataset to {file_path}")
    else:
        print(f"EmoBank dataset already exists at {file_path}")
    
    return file_path

# Download the dataset
emobank_path = download_emobank()

# Load the EmoBank dataset
df = pd.read_csv(emobank_path)

print(f"Dataset shape: {df.shape}")
print(f"\nColumn names: {list(df.columns)}")
print(f"\nFirst few rows:")
display(df.head())

# Check for missing values
print(f"\nMissing values:")
print(df.isnull().sum())

EmoBank dataset already exists at emobank.csv
Dataset shape: (10062, 6)

Column names: ['id', 'split', 'V', 'A', 'D', 'text']

First few rows:


,id,split,V,A,D,text
0,110CYL068_1036_1079,train,3.00,3.00,3.20,"Remember what she said in my last letter? """
1,110CYL068_1079_1110,test,2.80,3.10,2.80,If I wasn't working here.
2,110CYL068_1127_1130,train,3.00,3.00,3.00,".."""
3,110CYL068_1137_1188,train,3.44,3.00,3.22,Goodwill helps people get off of public assist...
4,110CYL068_1189_1328,train,3.55,3.27,3.46,Sherry learned through our Future Works class ...



Missing values:
id       0
split    0
V        0
A        0
D        0
text     1
dtype: int64


In [ ]:
import itertools
from scipy import stats

# Define a grid of bin sizes for each dimension
v_bin_options = [5, 10, 20, 30, 40, 50, 100, 250, 500, 750, 1000]
a_bin_options = [5, 10, 20, 30, 40, 50, 100, 250, 500, 750, 1000]
d_bin_options = [5, 10, 20, 30, 40, 50, 100, 250, 500, 750, 1000]
sample_size = 50

results = []

for v_bins, a_bins, d_bins in itertools.product(v_bin_options, a_bin_options, d_bin_options):
    df_test = df.copy()
    df_test['V_bin'] = pd.cut(df_test['V'], bins=v_bins, labels=False)
    df_test['A_bin'] = pd.cut(df_test['A'], bins=a_bins, labels=False)
    df_test['D_bin'] = pd.cut(df_test['D'], bins=d_bins, labels=False)
    df_test['strata'] = (
        df_test['V_bin'].astype(str) + '_' +
        df_test['A_bin'].astype(str) + '_' +
        df_test['D_bin'].astype(str)
    )
    strata_counts = df_test['strata'].value_counts()
    strata_proportions = strata_counts / len(df_test)
    sampled_dfs = []
    for stratum in strata_counts.index:
        stratum_data = df_test[df_test['strata'] == stratum]
        n_samples = max(1, int(strata_proportions[stratum] * sample_size))
        if len(stratum_data) >= n_samples:
            sampled_strata = stratum_data.sample(n=n_samples, random_state=42)
        else:
            sampled_strata = stratum_data
        sampled_dfs.append(sampled_strata)
    test_sample = pd.concat(sampled_dfs, ignore_index=True)
    if len(test_sample) > sample_size:
        test_sample = test_sample.sample(n=sample_size, random_state=42)
    # Calculate similarity metrics
    ks_pvalues = {}
    for dim in ['V', 'A', 'D']:
        ks_stat, ks_pvalue = stats.ks_2samp(df[dim], test_sample[dim])
        ks_pvalues[dim] = ks_pvalue
    results.append({
        'v_bins': v_bins,
        'a_bins': a_bins,
        'd_bins': d_bins,
        'ks_pvalue_V': ks_pvalues['V'],
        'ks_pvalue_A': ks_pvalues['A'],
        'ks_pvalue_D': ks_pvalues['D'],
    })

# Create DataFrame of results
results_df = pd.DataFrame(results)

# Filter for combinations where all p-values > 0.05
passing = results_df[
    (results_df['ks_pvalue_V'] > 0.05) &
    (results_df['ks_pvalue_A'] > 0.05) &
    (results_df['ks_pvalue_D'] > 0.05)
]

print("Bin size combinations with p-value > 0.05 for all dimensions:")
display(passing)

Bin size combinations with p-value > 0.05 for all dimensions:


,v_bins,a_bins,d_bins,ks_pvalue_V,ks_pvalue_A,ks_pvalue_D
260,20,10,250,0.099706,0.091109,0.105889
261,20,10,500,0.099706,0.091109,0.105889
262,20,10,750,0.099706,0.091109,0.105889
263,20,10,1000,0.099706,0.091109,0.105889
301,20,50,40,0.080028,0.103827,0.082041
557,40,100,250,0.146201,0.113198,0.263320
558,40,100,500,0.146201,0.113198,0.263320
559,40,100,750,0.146201,0.113198,0.263320
560,40,100,1000,0.146201,0.113198,0.263320
564,40,250,30,0.058496,0.105831,0.142904


In [ ]:
passing.to_csv('Data/bin_combinations.csv', index=False)

In [ ]:
# Slice the first row of every combination of V, A, and D p-values that are equal
first_rows = passing.drop_duplicates(subset=['ks_pvalue_V', 'ks_pvalue_A', 'ks_pvalue_D']).copy()

# Identify the minimnum p-value among the three and add it as a column
first_rows['min_pvalue'] = first_rows[['ks_pvalue_V', 'ks_pvalue_A', 'ks_pvalue_D']].min(axis=1)
first_rows.sort_values(by='min_pvalue', ascending=False, inplace=True)
first_rows

,v_bins,a_bins,d_bins,ks_pvalue_V,ks_pvalue_A,ks_pvalue_D,min_pvalue
894,250,40,30,0.220482,0.394363,0.607139,0.220482
1052,500,250,250,0.184615,0.567647,0.263320,0.184615
1061,500,500,50,0.482224,0.126144,0.237728,0.126144
557,40,100,250,0.146201,0.113198,0.263320,0.113198
1049,500,250,40,0.106355,0.221265,0.163611,0.106355
896,250,40,50,0.384705,0.105831,0.443996,0.105831
260,20,10,250,0.099706,0.091109,0.105889,0.091109
301,20,50,40,0.080028,0.103827,0.082041,0.080028
678,50,100,250,0.135100,0.079512,0.142904,0.079512
820,100,500,100,0.160288,0.072590,0.402543,0.072590


In [ ]:
# Find the row with the largest min_pvalue
best_config = first_rows.loc[first_rows['min_pvalue']==first_rows['min_pvalue'].max()].copy()
best_config.reset_index(drop=True, inplace=True)

print(f"Best configuration:")
print(f"V bins: {best_config['v_bins'][0]}")
print(f"A bins: {best_config['a_bins'][0]}")
print(f"D bins: {best_config['d_bins'][0]}")

# Generate sample using the best configuration
df_best = df.copy()
df_best['V_bin'] = pd.cut(df_best['V'], bins=best_config['v_bins'][0], labels=False)
df_best['A_bin'] = pd.cut(df_best['A'], bins=best_config['a_bins'][0], labels=False)
df_best['D_bin'] = pd.cut(df_best['D'], bins=best_config['d_bins'][0], labels=False)
df_best['strata'] = (
    df_best['V_bin'].astype(str) + '_' +
    df_best['A_bin'].astype(str) + '_' +
    df_best['D_bin'].astype(str)
)

# Sample from each stratum
strata_counts = df_best['strata'].value_counts()
strata_proportions = strata_counts / len(df_best)
sampled_dfs = []

for stratum in strata_counts.index:
    stratum_data = df_best[df_best['strata'] == stratum]
    n_samples = max(1, int(strata_proportions[stratum] * sample_size))
    
    if len(stratum_data) >= n_samples:
        sampled_strata = stratum_data.sample(n=n_samples, random_state=42)
    else:
        sampled_strata = stratum_data
    
    sampled_dfs.append(sampled_strata)

# Combine and limit to sample_size
best_sample = pd.concat(sampled_dfs, ignore_index=True)
if len(best_sample) > sample_size:
    best_sample = best_sample.sample(n=sample_size, random_state=42)

# Remove helper columns and keep only original columns
best_sample_clean = best_sample[['id', 'split', 'V', 'A', 'D', 'text']].reset_index(drop=True)

In [72]:
print(f"\nBest sample size: {len(best_sample_clean)}")
print(f"\nBest sample V, A, D statistics:")
print(best_sample_clean[['V', 'A', 'D']].describe())
print(df[['V', 'A', 'D']].describe())


Best sample size: 50

Best sample V, A, D statistics:
              V          A          D
count  50.00000  50.000000  50.000000
mean    2.93300   3.112600   3.067000
std     0.52632   0.320661   0.257121
min     1.57000   2.620000   2.570000
25%     2.61750   2.875000   2.902500
50%     2.95000   3.045000   3.000000
75%     3.27250   3.215000   3.215000
max     4.20000   4.200000   3.600000
                  V             A             D
count  10062.000000  10062.000000  10062.000000
mean       2.976690      3.041785      3.062763
std        0.348715      0.259541      0.209675
min        1.200000      1.800000      1.780000
25%        2.800000      2.890000      3.000000
50%        3.000000      3.000000      3.090000
75%        3.120000      3.200000      3.200000
max        4.600000      4.400000      4.200000


In [74]:
best_sample_clean.to_csv('Data/annotation_sample.csv', index=False)